# Nghiên cứu đối chứng đặc trưng (Ablation Study) — Logistic Regression cố định

**Người thực hiện:** Dương Gia Phát — 3124411212  
**Mô hình cố định:** Logistic Regression (L2 penalty, C=1.0, solver=lbfgs, max_iter=2000, tol=1e-4, seed=42)  
**Bộ chia dữ liệu:** 5-Fold Stratified Cross-Validation cố định trên 3.539 mẫu Train (`student_dropout_seed42_v1`)  
**Mục tiêu nghiên cứu:**
1. **Câu hỏi 1 (A–B):** Bỏ nhóm thông tin học kỳ 2 làm hiệu năng mô hình thay đổi như thế nào? Đóng góp thực tế của kết quả học tập HK2 đối với việc phát hiện sinh viên bỏ học?
2. **Câu hỏi 2 (C–A):** Việc bổ sung các đặc trưng mới (tỷ lệ qua môn HK1/HK2, chênh lệch điểm HK2–HK1) có cải thiện hiệu năng dự đoán và độ ổn định so với dữ liệu gốc không?


## 1. Thiết lập môi trường và đường dẫn

Đảm bảo mã nguồn nạp đúng các mô-đun dùng chung trong thư mục `src/`.

In [ ]:
import sys
from pathlib import Path
import json
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Thiết lập đường dẫn gốc dự án
NOTEBOOK_DIR = Path.cwd()
PROJECT_DIR = NOTEBOOK_DIR.parents[1] if NOTEBOOK_DIR.name == 'm05_logistic_regression' else Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.data import load_student_data, load_splits, LABELS
from src.features import FeatureEngineer, make_preprocessor, FEATURE_SET_A, FEATURE_SET_B, FEATURE_SET_C

print(f"Thư mục dự án: {PROJECT_DIR}")
print(f"Các nhãn phân loại: {LABELS}")


## 2. Dữ liệu và định nghĩa ba bộ đặc trưng A, B, C

- **Cấu hình A (Mốc đối chứng):** 36 đặc trưng gốc được kiểm chứng của bộ dữ liệu UCI 697.
- **Cấu hình B (Bỏ HK2):** 30 đặc trưng. Loại bỏ toàn bộ 6 thuộc tính của học kỳ 2 nhằm khảo sát kịch bản dự đoán trước khi có điểm HK2.
- **Cấu hình C (Bổ sung đặc trưng tạo mới):** 39 đặc trưng. Bổ sung 3 đặc trưng tỷ lệ qua môn và chênh lệch điểm giữa 2 học kỳ vào bộ đặc trưng gốc A.

In [ ]:
X, y = load_student_data()
train_ids, test_ids, folds, split_manifest = load_splits()
X_train, y_train = X.iloc[train_ids].copy(), y.iloc[train_ids].copy()

print(f"Kích thước tập Train: {X_train.shape[0]} mẫu, {X_train.shape[1]} đặc trưng gốc")
print(f"Phân phối lớp tập Train:")
for label in LABELS:
    print(f"  - {label}: {(y_train == label).sum()} mẫu ({(y_train == label).mean():.2%})")


## 3. Khám phá và kiểm chứng công thức đặc trưng tạo mới

Các đặc trưng mới trong Cấu hình C:
1. `Curricular units 1st sem (approved ratio)`: `approved / enrolled` (gán 0.0 khi `enrolled == 0`).
2. `Curricular units 2nd sem (approved ratio)`: `approved / enrolled` (gán 0.0 khi `enrolled == 0`).
3. `Curricular units grade difference (2nd - 1st)`: `grade_sem2 - grade_sem1`.

In [ ]:
fe = FeatureEngineer()
X_train_c = fe.transform(X_train)

new_cols = [
    "Curricular units 1st sem (approved ratio)",
    "Curricular units 2nd sem (approved ratio)",
    "Curricular units grade difference (2nd - 1st)"
]

print("Thống kê mô tả 3 đặc trưng tạo mới trên tập Train:")
display(X_train_c[new_cols].describe().round(4))


## 4. Tải kết quả thực nghiệm đã kiểm chứng

Đọc kết quả từ 3 lượt chạy đã lưu tại `experiments/runs/`:
- Cấu hình A: `m05_logistic_regression_v1_20261104T070444_33387318` (Lê Văn Hiếu)
- Cấu hình B: `m05_logistic_regression_ablation_b_20261104T104503_4007289c` (Dương Gia Phát)
- Cấu hình C: `m05_logistic_regression_ablation_c_20261104T104521_4cf44704` (Dương Gia Phát)

In [ ]:
ablation_df = pd.read_csv(PROJECT_DIR / "final/experiments/ablation.csv")
cols_display = [
    "configuration", "feature_set_name", "n_features", "macro_f1_mean", "macro_f1_std",
    "diff_macro_f1_vs_A", "accuracy_mean", "recall_dropout_mean", "recall_enrolled_mean",
    "recall_graduate_mean", "train_valid_gap_macro_f1"
]
print("BẢNG KẾT QUẢ ĐỐI CHỨNG ABLATION STUDY (5-FOLD CV):")
display(ablation_df[cols_display].round(4))


## 5. Chi tiết từng Fold kiểm định và đo đạc chênh lệch

Kiểm tra độ chênh lệch Macro-F1 theo từng fold cụ thể giữa A–B và C–A để đánh giá tính nhất quán.

In [ ]:
run_a_path = PROJECT_DIR / "final/experiments/runs/m05_logistic_regression_v1_20261104T070444_33387318"
run_b_path = PROJECT_DIR / "final/experiments/runs/m05_logistic_regression_ablation_b_20261104T104503_4007289c"
run_c_path = PROJECT_DIR / "final/experiments/runs/m05_logistic_regression_ablation_c_20261104T104521_4cf44704"

df_a = pd.read_csv(run_a_path / "fold_metrics.csv")
df_b = pd.read_csv(run_b_path / "fold_metrics.csv")
df_c = pd.read_csv(run_c_path / "fold_metrics.csv")

fold_summary = pd.DataFrame({
    "Fold": [1, 2, 3, 4, 5],
    "F1_B (No Sem2)": df_b["valid_f1_macro"],
    "F1_A (Baseline)": df_a["valid_f1_macro"],
    "F1_C (Engineered)": df_c["valid_f1_macro"],
    "Diff (A - B)": df_a["valid_f1_macro"] - df_b["valid_f1_macro"],
    "Diff (C - A)": df_c["valid_f1_macro"] - df_a["valid_f1_macro"],
})

print("CHÊNH LỆCH THEO TỪNG FOLD KIỂM ĐỊNH:")
display(fold_summary.round(4))

print(f"Chênh lệch trung bình A - B: {fold_summary['Diff (A - B)'].mean():.4f} (100% các fold A đều vượt B)")
print(f"Chênh lệch trung bình C - A: {fold_summary['Diff (C - A)'].mean():.4f} (4/5 fold C cải thiện so với A)")


## 6. Trực quan hóa kết quả đối chứng

Hiển thị 3 biểu đồ phân tích:
1. So sánh Macro-F1 trung bình có kèm khoảng độ lệch chuẩn.
2. So sánh Recall từng lớp để xem lớp nào hưởng lợi nhiều nhất.
3. Biểu đồ đường thể hiện độ ổn định qua từng fold.

In [ ]:
from IPython.display import Image, display

print("1. Biểu đồ so sánh Macro-F1 giữa A, B, C:")
display(Image(filename=str(PROJECT_DIR / "final/reports/figures/ablation_macro_f1_comparison.png")))

print("2. Biểu đồ so sánh Recall từng lớp:")
display(Image(filename=str(PROJECT_DIR / "final/reports/figures/ablation_class_recall_comparison.png")))

print("3. Biểu đồ độ ổn định qua 5 Fold:")
display(Image(filename=str(PROJECT_DIR / "final/reports/figures/ablation_fold_stability.png")))


## 7. Phân tích kết quả chuyên môn và kết luận

### 7.1. Phân tích tác động của nhóm thông tin Học kỳ 2 (A so với B)
- **Hiệu năng suy giảm mạnh khi thiếu thông tin HK2:** Macro-F1 giảm từ 0,6997 (Cấu hình A) xuống 0,6578 (Cấu hình B), tương ứng mức giảm 0,0420 điểm F1 (-6,0%). Accuracy giảm từ 77,54% xuống 74,62%.
- **Lớp chịu tác động nặng nề nhất là Enrolled:** Recall của lớp Enrolled giảm từ 37,80% xuống còn 29,92% (giảm 7,88 điểm phần trăm). Điều này cho thấy thông tin tích lũy và số môn qua ở HK2 là cơ sở then chốt để phân biệt sinh viên còn theo học với sinh viên đã bỏ học.
- **Độ ổn định:** Cấu hình B có độ lệch chuẩn qua 5 fold là 0,0149, cao hơn đáng kể so với Cấu hình A (0,0099). Thiếu thông tin học tập năm thứ nhất khiến mô hình nhạy cảm và dao động nhiều hơn giữa các tập con.
- **Giới hạn thời điểm dữ liệu:** Kết quả này khẳng định rõ ràng rằng một mô hình sử dụng toàn bộ 36 biến chỉ có thể được triển khai sau khi đã kết thúc học kỳ 2 (cuối năm thứ nhất). Nếu muốn cảnh báo sớm ngay khi sinh viên mới nhập học hoặc hết HK1, hệ thống bắt buộc phải chấp nhận mức hiệu năng thấp hơn tương đương Cấu hình B.

### 7.2. Phân tích giá trị của Feature Engineering (C so với A)
- **Mức cải thiện thực tế:** Việc bổ sung 3 đặc trưng tỷ lệ qua môn và chênh lệch điểm giúp Macro-F1 CV tăng từ 0,6997 lên 0,7031 (tăng +0,0033 điểm F1). Cải thiện xuất hiện tại 4 trên 5 fold kiểm định.
- **Tác động lên từng lớp:** Recall lớp Dropout tăng từ 77,48% lên 77,74% (+0,26%); Recall lớp Enrolled tăng từ 37,80% lên 38,43% (+0,63%); Recall lớp Graduate giữ mức cao 91,79%.
- **Đánh giá khách quan:** Với mô hình tuyến tính Logistic Regression, mức tăng +0,0033 là tương đối khiêm tốn. Nguyên nhân do mô hình đã có các biến số lượng môn đăng ký và qua môn dạng tuyến tính. Tuy nhiên, các đặc trưng tỷ lệ phi tuyến `approved / enrolled` và xu hướng thay đổi điểm số vẫn đóng góp tín hiệu bổ sung hữu ích, cải thiện trực tiếp khả năng nhận diện sinh viên có nguy cơ bỏ học mà không làm tăng nguy cơ quá khớp (train–validation gap ở Cấu hình C là 0,0563, hoàn toàn tương đương 0,0559 của Cấu hình A).
